<a href="https://colab.research.google.com/github/debo-ogunnowo/Prompt-Inference-System/blob/main/classifier_category_match_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import pipeline
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df1 = pd.read_csv('/content/drive/MyDrive/FYP/classifier_eval_labeled.csv')
df1.head()

,original prompt,reconstructed_prompt,original_category
0,What do you know about Korsakoff's syndrome,What is Korsakoff's syndrome and what are its ...,Research&Inquiry
1,"search engine indexing, how does that work beh...",explain how search engine indexing works behin...,Research&Inquiry
2,Compare the strengths and weaknesses of parlia...,Compare and contrast the parliamentary and pre...,Content Generation
3,how do you classify different types of energy ...,Classify different types of energy sources,Research&Inquiry
4,How do mangrove trees survive in saline coasta...,How do mangrove trees survive in saline coasta...,Research&Inquiry


In [ ]:
# Eval 2: Category Agreement
import torch

print("\n Evaluating category agreement...")
label_map = {
    'LABEL_0': 'Research&Inquiry',
    'LABEL_1': 'Content Generation',
    'LABEL_2': 'Text Refinement'
}

classifier = pipeline(
    'text-classification',
    model='/content/drive/MyDrive/prompt_classifier_v1',
    tokenizer='/content/drive/MyDrive/prompt_classifier_v1',
    device=0 if torch.cuda.is_available() else -1
)

def get_category(text):
  try:
    result = classifier(
      str(text),
      truncation=True,
      max_length=128
    )[0]
    return label_map[result['label']]
  except Exception as e:
    return 'Unknown'

df1['reconstructed_category'] = df1['reconstructed_prompt'].apply(get_category)
df1['category_match'] = (
    df1['original_category'] == df1['reconstructed_category']
)

agreement_rate = df1['category_match'].mean()
print(f"Overall category agreement rate: {agreement_rate:.4f}, {agreement_rate:.2%}")
print("\nAgreement rate by original category:")
print(df1.groupby('original_category')['category_match'].mean())



 Evaluating category agreement...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Overall category agreement rate: 0.7860, 78.60%

Agreement rate by original category:
original_category
Content Generation    0.782609
Research&Inquiry      0.860000
Text Refinement       0.666667
Name: category_match, dtype: float64


In [ ]:
df1.head()

,original prompt,reconstructed_prompt,original_category,reconstructed_category,category_match
0,What do you know about Korsakoff's syndrome,What is Korsakoff's syndrome and what are its ...,Research&Inquiry,Research&Inquiry,True
1,"search engine indexing, how does that work beh...",explain how search engine indexing works behin...,Research&Inquiry,Research&Inquiry,True
2,Compare the strengths and weaknesses of parlia...,Compare and contrast the parliamentary and pre...,Content Generation,Content Generation,True
3,how do you classify different types of energy ...,Classify different types of energy sources,Research&Inquiry,Content Generation,False
4,How do mangrove trees survive in saline coasta...,How do mangrove trees survive in saline coasta...,Research&Inquiry,Research&Inquiry,True


In [ ]:
df1.to_csv('/content/drive/MyDrive/FYP/prompt_classifier_v1_recon_results.csv')